# pieces
**Depends on:** 02_food, 03_board, HexMagic.game.flag, HexMagic.styles

Pieces, squads, and roster management. First notebook that introduces game entities.

In [ ]:
#| default_exp magic/pieces

In [ ]:
#| export
from __future__ import annotations
from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
import io, base64, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import NamedTuple
from enum import Enum
from collections import defaultdict
import random
import math
import uuid
import httpx

from monsterui.all import *
from fastcore.basics import patch

In [ ]:
#| export
from fasthtml.common import *
from fasthtml.jupyter import *
from fasthtml.common import *

In [ ]:
#| export
from HexMagic.geology import Geology
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import Terrain, HexGrid, HexPosition
from HexMagic.overlay import CreamOverlay, RiverOverlay,OceanWaveOverlay
from HexMagic.game.flag import CountryFlag , PieceType, GameContext, DiagramGlyphs
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.terrainpatterns import TerrainPatterns
from HexMagic.climate import TerrainFactory
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.primitives import MapRect, MapPath,MapCord,  MapSize, HexRegion


In [ ]:
#| export
from HexMagic.magic.board import GameParts,FogOverlay, MapPlace
from HexMagic.magic.food import FOOD_TABLE, FoodStats, OasisOverlay, HexNumberOverlay

In [ ]:
showDemo = False

## Piece

In [ ]:
#| export
@dataclass
class Piece:
    id:str = ""
    flag:CountryFlag | None = None
    type:PieceType = PieceType.PAWN
    place:MapPlace | None = None
    timeout:int = 0
    name:str = ""
    year:int = 1900
    animal: str = ""
    countryName: str = ""
    lantern = StyleCSS("blank",fill="#007",opacity=0.3)

In [ ]:
#| export
@patch
def __eq__(self: Piece, other) -> bool:
    if not isinstance(other, Piece): return False
    return self.id == other.id

@patch
def __hash__(self: Piece) -> int:
    return hash(self.id)


In [ ]:
#| export
@patch(as_prop=True)
def food(self: Piece) -> FoodStats:
    return FOOD_TABLE[self.type]


In [ ]:
#| export
@patch
def sight(self: Piece) -> HexRegion:
    """All hex indices visible from this piece's location."""
    bonus = self.food.bonus_sight
    effective_sight = max(1, self.place.level) + bonus
    
    return HexRegion(
        hexes=set([self.place.parent.hp2i(hp) 
                   for hp in self.place.location.field_of_view(self.place.facing, effective_sight)]),
        hexGrid=self.place.parent.grid)


### Graphics

In [ ]:
#| export


@patch
def fetch_avatar(self: Piece, style: str = "avataaars") -> str:
    """Fetch and cache DiceBear SVG avatar using piece name as seed."""
    if not hasattr(self, '_avatar_svg') or self._avatar_svg is None:
        seed = self.name or f"piece_{id(self)}"
        resp = httpx.get(f"https://api.dicebear.com/9.x/{style}/svg?seed={seed}")
        resp.raise_for_status()
        self._avatar_svg = resp.text
    return self._avatar_svg

@patch
def avatar_id(self: Piece) -> str:
    return f"avatar_{self.name or id(self)}"

@patch
def register_avatar(self: Piece, builder: SVGBuilder, style: str = "avataaars"):
    """Register avatar as a <symbol> definition on the builder (idempotent)."""
    aid = self.avatar_id()
    # Skip if already registered
    if any(getattr(d, 'attributes', {}).get('id') == aid for d in builder.definitions):
        return
    svg_raw = self.fetch_avatar(style)
    # Extract viewBox from the SVG
    
    vb_match = re.search(r'viewBox="([^"]+)"', svg_raw)
    vb = vb_match.group(1) if vb_match else "0 0 280 280"
    # Strip outer <svg> tags to get inner content
    inner = re.sub(r'<svg[^>]*>', '', svg_raw)
    inner = re.sub(r'</svg>\s*$', '', inner).strip()
    symbol = SVGDef("symbol", aid, inner, viewBox=vb)
    builder.add_definition(symbol)

@patch
def draw_avatar(self: Piece, center: MapCord, builder: SVGBuilder,
                size: float = 40, layer: str = None,
                opacity: float = 1.0, attrs: dict = None) -> str:
    """Render avatar at center with given size. Registers symbol if needed."""
    self.register_avatar(builder)
    aid = self.avatar_id()
    x, y = center.x - size / 2, center.y - size / 2
    extra = CountryFlag._render_attrs(attrs) if attrs else ""
    svg = (f'<use href="#{aid}" x="{x:.1f}" y="{y:.1f}" '
           f'width="{size:.1f}" height="{size:.1f}"{extra}/>')
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer:
        builder.adjust(layer, svg)
    return svg


In [ ]:
#| export
@patch
def icon(piece: Piece, size=24):
    """Render a tiny inline SVG of a chess piece using flag colors."""
    b = SVGBuilder()
    b.width, b.height = size, size
    scale = size / 45.0
    piece.flag.draw_piece(
        piece.type, MapCord(size/2, size/2), b,
        scale=scale, size='board', layer="icon",
        piece_id=f"ico_{piece.id[:8]}")
    return Div(NotStr(b.xml()))



@patch
def __ft__(self: Piece):
    b = SVGBuilder()
    b.width, b.height = 60, 60
    self.draw_avatar(MapCord(30, 30), b, size=50, layer="avatar")

    pip = self.icon(size=20)
    pip_flipped = Div(self.icon(size=20), style="transform: rotate(180deg);")

    return Card(
        Div(
            # Top-left pip
            Div(pip, style="position:absolute; top:6px; left:6px;"),
            # Bottom-right pip
            Div(pip, style="position:absolute; bottom:6px; right:6px;"),
            # Center content
            DivCentered(
                NotStr(b.xml()),
                P(str(self.year), cls=TextPresets.muted_sm),
            ),
            style="position:relative; min-height:100px;"
        ),
        cls=CardT.hover
    )


In [ ]:
@patch
def __ft__(self: Piece):
    if self.place is not None and self.place.parent:
        parts = self.place.parent
        overlays = (
            CreamOverlay(stylized=True),
            RiverOverlay(max_width=4),
            FogOverlay(),
            HexNumberOverlay(),
        )
        middle = self.place.place_ft(self.place, *overlays, distance=1,radius=20)

    else:

        b = SVGBuilder()
        b.width, b.height = 60, 60
        self.draw_avatar(MapCord(30, 30), b, size=50, layer="avatar")
        middle = NotStr(b.xml())

    pip = self.icon(size=20)
    pip_flipped = Div(self.icon(size=20), style="transform: rotate(180deg);")

    grid_style = f"""
        display: grid;
        grid-template-columns: 24px 1fr 24px;
        grid-template-rows: 24px 1fr 24px;
        gap: 4px;
        min-height: 120px;
        background-color: {self.flag.lightComp};
        overflow: hidden;
    """

    return Card(
        Div(
            # Top-left pip
            Div(pip, style="grid-column:1; grid-row:1;"),
            #Div(P(self.name, cls=TextPresets.muted_sm), style="grid-column:2; grid-row:1;"),
            # Center: image + name span middle column all rows
            DivCentered(
                middle,
                P(self.name, cls=TextPresets.muted_sm),
                style="grid-column:2; grid-row:1/4; flex-direction:column;"
            ),
            # Bottom-right pip
            Div(pip_flipped, style="grid-column:3; grid-row:3;"),
            style=grid_style
        ),
        cls=CardT.hover
    )


can you write it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
@patch
def __ft__(self: Piece):
    b = SVGBuilder()
    b.width, b.height = 60, 60
    self.draw_avatar(MapCord(30, 30), b, size=50, layer="avatar")

    pip = self.icon(size=20)
    pip_flipped = Div(self.icon(size=20), style="transform: rotate(180deg);")

    return Card(
        Div(
            # Top-left pip
            Div(pip, style="position:absolute; top:6px; left:6px;"),
            # Bottom-right pip
            Div(pip_flipped, style="position:absolute; bottom:6px; right:6px;"),
            # Center content
            DivCentered(
                NotStr(b.xml()),
                P(str(self.year), cls=TextPresets.muted_sm),
            ),
            style="position:relative; min-height:100px;"
        ),
        cls=CardT.hover
    )
```

This gives you the card-style layout with pips in opposite corners. The food bar can drop in where the year `P` is once you've sorted out the data model. Give it a try and let me know how it renders!

can we do this as a table instead
```
pip | |
    | image |
    | name | pip
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Sure! Just to confirm I'm reading your layout right:

| col 1 | col 2 | col 3 |
|-------|-------|-------|
| pip   |       |       |
|       | image |       |
|       | name  | pip   |

Is that right? And do you want a real HTML `<table>`, or a CSS grid/flexbox approach (which is generally more flexible for this kind of card layout)?

I think more of the flex box. this way the image in the center doesn't conflict with the pips

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good call — a CSS grid with 3 columns × 3 rows would handle this cleanly. Something like:

- top-left cell: pip
- middle column (rows 1–3): image + name centered
- bottom-right cell: pip

Does that match your vision, or do you want the name *below* the grid entirely (outside the 3-col structure)?

perfect. can you write it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
@patch
def __ft__(self: Piece):
    b = SVGBuilder()
    b.width, b.height = 60, 60
    self.draw_avatar(MapCord(30, 30), b, size=50, layer="avatar")

    pip = self.icon(size=20)
    pip_flipped = Div(self.icon(size=20), style="transform: rotate(180deg);")

    grid_style = """
        display: grid;
        grid-template-columns: 24px 1fr 24px;
        grid-template-rows: 24px 1fr 24px;
        gap: 4px;
        min-height: 120px;
    """

    return Card(
        Div(
            # Top-left pip
            Div(pip, style="grid-column:1; grid-row:1;"),
            Div(P(self.name, cls=TextPresets.muted_sm), style="grid-column:2; grid-row:1;"),
            # Center: image + name span middle column all rows
            DivCentered(
                NotStr(b.xml()),
                
                style="grid-column:2; grid-row:1/4; flex-direction:column;"
            ),
            # Bottom-right pip
            Div(pip_flipped, style="grid-column:3; grid-row:3;"),
            style=grid_style
        ),
        cls=CardT.hover
    )
```

The key idea: middle column spans all 3 rows so the image/name area is never crowded by the pips. Give it a run and let me know how it looks!

## Target

In [ ]:
#| export
class TargetType(Enum):
    KING = "king"
    OASIS = "oasis"
    CONSOLIDATE = "consolidate"

In [ ]:
#| export
@dataclass
class SquadTarget:
    type: TargetType
    hex_idx: int = -1         # -1 for CONSOLIDATE
    value: float = 0.0
    enemy_squad: str = ""
    turns_remaining: int = 5  # countdown within strategy cycle
    
    @property
    def label(self) -> str:
        if self.type == TargetType.CONSOLIDATE:
            return f"🏰 consolidate ({self.turns_remaining}t left)"
        if self.type == TargetType.KING:
            return f"⚔️ {self.enemy_squad} king @ {self.hex_idx} ({self.turns_remaining}t)"
        return f"🌾 oasis @ {self.hex_idx} ({self.turns_remaining}t)"
    
    @property
    def is_expired(self) -> bool:
        return self.turns_remaining <= 0
    
    def tick(self):
        self.turns_remaining = max(0, self.turns_remaining - 1)


## Squad

In [ ]:
#| export
class PlanContext(NamedTuple):
    pantry: int
    placed: int
    threat: float
    stance: str

    def has_drifted(self, other: 'PlanContext',
                    pantry_tol: int = 20,
                    placed_tol: int = 1,
                    threat_tol: float = 2.0) -> bool:
        return (
            abs(self.pantry - other.pantry) > pantry_tol
            or abs(self.placed - other.placed) > placed_tol
            or abs(self.threat - other.threat) > threat_tol
            or self.stance != other.stance
        )


In [ ]:
#| export
@dataclass
class PlacementStrategy:
    """Squad personality that shapes piece priority and stance thresholds."""
    name: str
    # Stance thresholds (how easily this personality shifts)
    defend_threat: int = 2       # enemy pieces near king to trigger defend
    attack_pantry: int = 60      # min king pantry to go offensive
    attack_placed: int = 4       # min pieces on board before attacking
    # Knight eagerness: lower = fires more readily
    knight_roi_mult: float = 1.0  # multiplied into knight ROI gate
    # NEW: spending discipline
    min_reserve_per_piece: int = 5    # pantry reserve per placed piece
    min_budget_ratio: float = 1.0     # won't buy unless income/upkeep >= this
    max_deficit_turns: int = 2        # skip buying if net negative for N+ turns
    buy_cooldown: int = 0             # min turns between purchases (0=every turn)
    afforable_min = 3                 # min number of pieces before we check on afford
    target_composition: dict[PieceType, int] = field(default_factory=dict)
    graduation: str = "balanced"

    @property
    def min_pawns(self: PlacementStrategy) -> int:
        """Pawn count from target_composition, with a sensible floor."""
        return max(2, self.target_composition.get(PieceType.PAWN, 2))
  # how many of each type to aim for

In [ ]:
#| export
class QueueEntry(NamedTuple):
    priority: float      # negated so heapq (min-heap) gives highest first
    seq: int          # insertion order tiebreak — avoids Enum comparison
    piece_type: PieceType


In [ ]:
#| export
@patch
def __lt__(self: QueueEntry, other: QueueEntry) -> bool:
    if self.priority != other.priority:
        return self.priority < other.priority
    return PURCHASE_COST[self.piece_type] < PURCHASE_COST[other.piece_type]


In [ ]:
#| export
PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
    min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0,
    target_composition={PieceType.BISHOP: 4, PieceType.PAWN: 0})

PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3,
    knight_roi_mult=0.7, min_reserve_per_piece=3, min_budget_ratio=0.6, buy_cooldown=0,
    target_composition={PieceType.KNIGHT: 2, PieceType.BISHOP: 1})

PlacementStrategy.BALANCED = PlacementStrategy(
    "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
    min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=1,
    target_composition={PieceType.BISHOP: 2, PieceType.KNIGHT: 1, PieceType.QUEEN: 1})

PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle", defend_threat=1, attack_pantry=999, attack_placed=99,
    min_reserve_per_piece=8, min_budget_ratio=1.2, buy_cooldown=2,
    target_composition={PieceType.BISHOP: 3, PieceType.PAWN: 0})


In [ ]:
#| export
@dataclass
class Squad:
    pieces: list[Piece]          = field(default_factory=list)
    name: str                    = ""
    year: int                    = 1900
    animal: str                  = ""
    flag: CountryFlag | None     = None
    countryName: str             = ""
    storage: int                 = 100
    queue: list[QueueEntry] = field(default_factory=list)  # heapq
    target: SquadTarget          = field(default_factory=lambda: SquadTarget(TargetType.CONSOLIDATE))
    strategy: PlacementStrategy  = field(default_factory=lambda: PlacementStrategy.TURTLE)
    plan_ctx: PlanContext | None = None   # conditions when queue was last built

    #alive:bool = True

    @classmethod
    def squads(cls, count: int = 4, flag: CountryFlag | None = None) -> list['Squad']:
        if flag is None:
            flag = CountryFlag.seaborn("husl", 1)[0]
        return [Squad(name=name, animal=animal, flag=flag) 
                for animal, name in flag.country_squads(count)]

In [ ]:
#| export
@patch
def kill(self: Squad):
    """Remove all pieces from the board. Called when king dies."""
    for p in self.pieces:
        p.place = None

In [ ]:
#| export
@patch
def king_alive(self: Squad) -> bool:
    """Is the king still on the board?"""
    return any(p.type == PieceType.KING and p.place is not None for p in self.pieces)

In [ ]:
#| export

@patch
def tick_lifespan(self: Squad) -> list[Piece]:
    """Increment tenure for placed pieces. Evict any that exceed lifespan."""
    evicted = []
    for p in self.pieces:
        if p.place is None or p.type == PieceType.KING:
            continue
        p.place.tenure += 1
        if p.place.tenure >= p.food.lifespan:
            p.place = None
            evicted.append(p)
    return evicted


tick_lifespand needs to use p.food.lifespan instead of LIFESPAN. Can you rewrite?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
@patch
def tick_lifespan(self: Squad) -> list[Piece]:
    """Increment tenure for placed pieces. Evict any that exceed lifespan."""
    evicted = []
    for p in self.pieces:
        if p.place is None or p.type == PieceType.KING:
            continue
        p.place.tenure += 1
        if p.place.tenure >= p.food.lifespan:
            p.place = None
            evicted.append(p)
    return evicted
```

In [ ]:
#| export
@patch
def make_squad(self: CountryFlag, piece_types: list[PieceType], year: int, seed: int = 0) -> Squad:
    """Create a Squad with animal name and age-appropriate named Pieces."""
    animal = self.animal_name()
    squad_nm = self.squad_name(animal, seed)
    
    rng = random.Random(hash((self.name, year, seed)))
    
    pieces = []
    for pt in piece_types:
        birth_year = year + rng.randint(20, 38)
        names = self.commonNames(year=birth_year)
        name = rng.choice(names)
        player = Piece(flag=self, type=pt, name=name, year=birth_year)
        player.id = player.name + "_" + uuid.uuid4().hex[:10]
        player.lantern = StyleCSS(f"{player.id}_lantern", fill=self.primary, opacity=0.3)

        player.animal = animal
        player.countryName = squad_nm

        pieces.append(player)

    
    return Squad(pieces=pieces, name=squad_nm, year=year, animal=animal, flag=self)


### ui

In [ ]:
#| export
@patch
def table(self: Squad):
    rows = []
    for p in self.pieces:
        loc = ""
        facing = ""
        if p.place and p.place.parent:
            loc = str(p.place.parent.hp2i(p.place.location))
            facing = str(p.place.facing.label)
        rows.append(Tr(
            Td(p.icon()),
            Td(p.name, cls=TextT.medium),
            Td(str(p.year), cls=TextPresets.muted_sm),
            Td(loc, cls=TextPresets.muted_sm),
            Td(facing, cls=TextPresets.muted_sm),
        ))
    
    return Card(
        Table(
            Thead(Tr(Th(""), Th("Name"), Th("Born"), Th("Loc"), Th("Facing"))),
            Tbody(*rows),
        ),
        header=(H4(self.name), P(f"🐾 {self.animal} · {len(self.pieces)} pieces", cls=TextPresets.muted_sm)),
    )


#| export
@patch
def __ft__(self: Squad):
    # Header: flag banner + squad name + animal
    flag_builder = self.flag.flag_banner(width=210, height=120, wavy=True)

    # Animal icon
    ab = SVGBuilder()
    ab.width, ab.height = 250,250
    ab.adjust("animal", self.flag.animal_svg(
        size='large', animal=self.animal,
        center=MapCord(125, 125),
        piece_id=f"sq_{self.name}"))


    header = DivFullySpaced(
       # NotStr(flag_builder.xml()),
        H3(self.name),
        NotStr(ab.xml()),
        

        cls=TextPresets.muted_sm)

    # Group pieces by type
    
    by_type = defaultdict(list)
    for p in self.pieces:
        by_type[p.type].append(p)

    sections = []
    for pt, pieces in by_type.items():
        # Section header with piece icon
        sec_header = DivLAligned(
            Span(pt.icon, cls=TextT.lg),
            H4(pt.value.title()),
            P(f"× {len(pieces)}", cls=TextPresets.muted_sm))
        # 4-wide grid of piece cards
        grid = Grid(*[p.__ft__() for p in pieces],
                    cols_sm=2, cols_md=4)
        sections.append(Div(sec_header, grid, cls='space-y-2'))

    return DivCentered(header,Card( *sections, cls='space-y-4'))


In [ ]:
#| export
@patch
def __ft__(self: Squad):
    # Header: flag banner + squad name + animal
    #ab = Div(NotStr(self.flag.logo(animal=self.animal, size= 65)))
    header = "" #DivFullySpaced(H3(self.name), ab, cls=TextPresets.muted_sm)

    for piece in self.pieces:
        piece.animal =  self.animal

    by_type = defaultdict(list)
    for p in self.pieces:
        by_type[p.type].append(p)

    sections = []
    for pt, pieces in by_type.items():
        sec_header = DivLAligned(
            Span(pt.icon, cls=TextT.lg),
            H4(pt.value.title()),
            P(f"× {len(pieces)}", cls=TextPresets.muted_sm),
            style="overflow-x:auto; padding-top:40px;")

        slides = [
            Div(p.__ft__(), cls="flex-shrink:0;")
            for p in pieces
        ]
        carousel = Div(
            *slides,
            style="display:flex; flex-direction:row; overflow-x:auto; gap:16px; padding:40px 16px 16px 40px;"
        )

        sections.append(Div(sec_header, carousel, cls='space-y-2'))

    return DivCentered(header, Card(*sections, cls='space-y-4'))


In [ ]:
#| export
@patch
def visible_allies(self: Piece, squad: Squad) -> list['Piece']:
    """Allies in this piece's sight cone, sorted by birth year then name."""
    if self.place is None:
        return []
    my_cone = self.sight().hexes
    parts = self.place.parent
    allies = []
    for p in squad.pieces:
        if p is self or p.place is None:
            continue
        idx = parts.hp2i(p.place.location)
        if idx in my_cone:
            allies.append(p)
    # Oldest first (lowest year), then alphabetically
    allies.sort(key=lambda p: (p.year, p.name))
    return allies


### Squad and GameParts

In [ ]:
#| export
@patch
def loadSquads(self:GameParts, year: int=1880, seed: int = 0,levels=3):
    self.flags = CountryFlag.seaborn("bright",levels=levels)
    oasis_idxs = self.fy.find_oasis(n=levels)
    self.squads = []
    for i,flag in enumerate(self.flags):
        roster = {
            PieceType.PAWN: 3, 
            #PieceType.BISHOP: 2, 
            #PieceType.QUEEN: 2, 
            PieceType.KING: 1, 
            #PieceType.KNIGHT: 4
        }
        pieceKinds = [pt for pt, n in roster.items() for _ in range(n)]


        squad = flag.make_squad(pieceKinds, year=year, seed =seed) 
        kingPlace = self.location_by_index(oasis_idxs[i])
        squad.pieces[-1].place = kingPlace
        self.squads.append(squad)

In [ ]:
MapPlace??

In [ ]:
myStuff = GameParts()

#| export
@patch
def __ft__(self: Piece):
    b = SVGBuilder()
    b.width, b.height = 120, 120
    self.draw_avatar(MapCord(60, 60), b, size=110, layer="avatar")

    # Piece icon (pip) and flag banner
    pip = self.icon(size=48)
    pip = Div(NotStr(self.flag.pieceLogo(piece_type=self.type,size=48)))
    #flag_b = self.flag.flag_banner(width=24, height=24, wavy=False)

    flag_b = self.flag.logo(
        size=48, animal=self.animal,
        piece_id=f"sq_{self.name}")


    primary = self.flag.primary
    secondary = self.flag.comp
    light = self.flag.lightComp

    # Year badge at bottom-center of image area
    year_badge = Div(
        str(self.year),
        style=f"""
            position:absolute; bottom:0; left:50%; transform:translate(-50%,50%);
            width:40px; height:40px; border-radius:12px;
            background:linear-gradient(to bottom,{light},{secondary});
            color:white; font-weight:800; font-size:14px;
            display:flex; align-items:center; justify-content:center;
            z-index:10;
        """)

    # Top-left circle with piece icon
    tl_circle = Div(
        Div(pip, style="position:absolute; right:8px; bottom:8px;"),
        style=f"""
            position:absolute; top:0; left:0; transform:translate(-50%,-50%);
            width:64px; height:64px; border-radius:50%;
            border:1px solid #e5e5e5; background:white;
            z-index:10;
        """)

    # Top-right circle with flag
    tr_circle = Div(
        Div(NotStr(flag_b), style="position:absolute; left:8px; bottom:8px;"),
        style=f"""
            position:absolute; top:0; right:0; transform:translate(50%,-50%);
            width:64px; height:64px; border-radius:50%;
            border:1px solid #e5e5e5; background:white;
            z-index:10;
        """)

    # Faded name behind avatar
    bg_text = Div(
        Div(self.type.value.title()),
        Div(self.name.split()[0] if self.name else ""),
        
        style=f"""
            position:absolute; top:20px; left:50%; transform:translateX(-50%);
            text-align:center; font-size:48px; line-height:0.85em;
            font-weight:800; text-transform:uppercase; font-style:italic;
            color:white; opacity:0.35; mix-blend-mode:overlay;
            pointer-events:none; z-index:1; letter-spacing:-0.05em;
        """)

    # Image area with gradient + chevron clip
    image_area = Div(
        bg_text,
        Div(NotStr(b.xml()), style="position:absolute; bottom:10px; left:50%; transform:translateX(-50%); z-index:2;"),
        style=f"""
            position:relative; height:220px; overflow:hidden;
            background:linear-gradient(to bottom,{self.flag.lightPrimary},{self.flag.darkPrimary});
            clip-path:polygon(0 0, 100% 0, 100% 90%, 50% 100%, 0 90%);
        """)
    #zDiv(self.countryName),
    # Inner card: image + badges + name
    inner = Div(
        Div(
            image_area, year_badge, tl_circle, tr_circle,
            style="position:relative; overflow:visible; padding-bottom:16px;"
        ),
        Div(
            H4(self.name, style="margin:0; font-size:18px; font-weight:700; letter-spacing:-0.02em;"),
            P(self.countryName, cls=TextPresets.muted_sm),
            style="text-align:center; padding:16px 0 8px 0;"
        ),
        style=f"""
            border-radius:20px; background:white; padding:16px;
            box-shadow:0 0 0 1px #e5e5e5;
        """)

    # Stats row
    stats = [
        ("Food", str(self.food.diet)),
        ("Sight", str(self.food.bonus_sight)),
        ("Moves", str(self.food.drain)),
    ]
    stat_divs = [Div(
        Div(val, style="font-size:14px; font-weight:700; margin-bottom:4px;"),
        Div(label, style="font-size:10px; text-transform:uppercase;"),
        style="text-align:center; padding:0 20px; color:white;"
    ) for label, val in stats]

    stats_row = Div(
        *stat_divs,
        style="display:flex; justify-content:center; gap:0; padding:16px 0; "
              "divide-x divide-gray-200;"
    )

    # Outer card
    light = self.flag.tri1
    return Div(
        Div(inner, stats_row,
            style=f"border-radius:24px; border:1px solid #e5e5e5; background:{light};"),
        style="width:280px; padding:0 20px; margin:8px auto;"
    )


So in theory if a piece has a MapPlace it could do an place_ft of it. I think this would be a more useful middle section of the piece.__ft__

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good idea — show the map view when placed, fall back to the avatar when not. Here's the updated `__ft__`:

```python
@patch
def __ft__(self: Piece):
    # Middle section: map view if placed, avatar otherwise
    if self.place is not None and self.place.parent is not None:
        middle = self.place.parent.place_ft(self.place)
    else:
        b = SVGBuilder()
        b.width, b.height = 120, 120
        self.draw_avatar(MapCord(60, 60), b, size=110, layer="avatar")
        middle = NotStr(b.xml())

    pip = Div(NotStr(self.flag.pieceLogo(piece_type=self.type, size=48)))
    flag_b = self.flag.logo(size=48, animal=self.animal, piece_id=f"sq_{self.name}")

    grid_style = """
        display: grid;
        grid-template-columns: 24px 1fr 24px;
        grid-template-rows: 24px 1fr 24px;
        gap: 4px;
        min-height: 120px;
    """

    return Card(
        Div(
            # Top-left pip
            Div(pip, style="grid-column:1; grid-row:1;"),
            # Top-right flag logo
            Div(NotStr(flag_b), style="grid-column:3; grid-row:1;"),
            # Center: map or avatar + name, spans middle column all rows
            DivCentered(
                middle,
                P(self.name, cls=TextPresets.muted_sm),
                style="grid-column:2; grid-row:1/4; flex-direction:column;"
            ),
            # Bottom-right pip (flipped)
            Div(pip, style="grid-column:3; grid-row:3; transform:rotate(180deg);"),
            style=grid_style
        ),
        cls=CardT.hover
    )
```

The nice thing here is `place_ft` already renders the facing bar, harvest stars, and pantry — so you get all that context for free in the card. Unplaced pieces still show their avatar.

Should this be called inside the piece _ft_ we are losing the corner pips which I liked

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're right — the pips are still in the code but let me make sure they match your original layout exactly (top-left and bottom-right only, no flag logo cluttering the corners):

```python
@patch
def __ft__(self: Piece):
    # Middle section: map view if placed, avatar otherwise
    if self.place is not None and self.place.parent is not None:
        middle = self.place.parent.place_ft(self.place)
    else:
        b = SVGBuilder()
        b.width, b.height = 120, 120
        self.draw_avatar(MapCord(60, 60), b, size=110, layer="avatar")
        middle = NotStr(b.xml())

    pip = Div(NotStr(self.flag.pieceLogo(piece_type=self.type, size=20)))
    pip_flipped = Div(NotStr(self.flag.pieceLogo(piece_type=self.type, size=20)),
                      style="transform:rotate(180deg);")

    grid_style = """
        display: grid;
        grid-template-columns: 28px 1fr 28px;
        grid-template-rows: 28px 1fr 28px;
        gap: 4px;
        min-height: 120px;
    """

    return Card(
        Div(
            Div(pip,         style="grid-column:1; grid-row:1;"),
            DivCentered(
                middle,
                P(self.name, cls=TextPresets.muted_sm),
                style="grid-column:2; grid-row:1/4; flex-direction:column;"
            ),
            Div(pip_flipped, style="grid-column:3; grid-row:3;"),
            style=grid_style
        ),
        cls=CardT.hover
    )
```

So yes — `place_ft` lives inside `__ft__`, the map appears when the piece is placed, avatar when not, and the corner pips are preserved as intended.

@patch
def __ft__(self: Piece):
    # Middle section: map view if placed, avatar otherwise
    if self.place is not None and self.place.parent is not None:
        middle = self.place.parent.place_ft(self.place)
    else:
        b = SVGBuilder()
        b.width, b.height = 120, 120
        self.draw_avatar(MapCord(60, 60), b, size=110, layer="avatar")
        middle = NotStr(b.xml())

    pip = Div(NotStr(self.flag.pieceLogo(piece_type=self.type, size=20)))
    pip_flipped = Div(NotStr(self.flag.pieceLogo(piece_type=self.type, size=20)),
                      style="transform:rotate(180deg);")

    grid_style = """
        display: grid;
        grid-template-columns: 28px 1fr 28px;
        grid-template-rows: 28px 1fr 28px;
        gap: 4px;
        min-height: 120px;
    """

    return Card(
        Div(
            Div(pip,         style="grid-column:1; grid-row:1;"),
            DivCentered(
                middle,
                P(self.name, cls=TextPresets.muted_sm),
                style="grid-column:2; grid-row:1/4; flex-direction:column;"
            ),
            Div(pip_flipped, style="grid-column:3; grid-row:3;"),
            style=grid_style
        ),
        cls=CardT.hover
    )


In [ ]:
myStuff.loadSquads()

In [ ]:
??FoodStats

In [ ]:
def PieceOverlay(**kw) -> OverlaySpec:
    """Render all pieces using their flag's chess-piece SVG."""
    def render(ctx) -> str:
        if not ctx.pieces:
            return ""

        grid = ctx.grid
        N = len(grid.hexes)
        scale = grid.radius * 0.8 / 22.5
        parts = []

        for i, piece in enumerate(ctx.pieces):
            if piece.place is None or piece.flag is None:
                continue

            coarse_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(coarse_idx):
                if 0 <= fi < N:
                    center = grid.hexes[fi].center
                    pid = f"pc_{piece.id}_{i}"

                    svg_str, pat_def = piece.flag.piece_svg(
                        piece.type,
                        MapCord(center.x, center.y),
                        scale=scale,
                        size='board',
                        piece_id=pid,
                    )

                    if pat_def is not None:
                        ctx.builder.add_definition(pat_def)

                    parts.append(svg_str)

        return '\n'.join(parts)

    return OverlaySpec("pieces", render, requires={'pieces'}, priority=80)


In [ ]:
#| export
def LanternOverlay(**kw) -> OverlaySpec:
    """Render sight regions for placed pieces using their lantern style, plus facing bars."""
    def render(ctx) -> str:
        grid, builder = ctx.grid, ctx.builder
        N = len(grid.hexes)
        r = grid.radius
        parts = []

        for piece in ctx.pieces:
            if piece.place is None: continue
            builder.add_style(piece.lantern)

            # Sight region
            for idx in sorted(piece.sight().hexes):
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        h = grid.hexes[fi]
                        parts.append(Hex(h.radius, h.center, piece.lantern, v=h.v).svg())

            # Facing bar at piece's location
            loc_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(loc_idx):
                if 0 <= fi < N:
                    c = grid.hexes[fi].center
                    glyphs = DiagramGlyphs(piece.flag, size=r * 0.7)
                    glyphs.register_styles(builder)
                    dirs = HexPosition.directions()
                    facing_dir = dirs.index(piece.place.facing) if piece.place.facing in dirs else 0
                    parts.append(glyphs.facing_bar(c, facing_dir))

        return '\n'.join(parts)

    return OverlaySpec("lanterns", render, requires={'pieces'}, priority=62)



In [ ]:
#| export
@patch
def __ft__(self: Piece):
    b = SVGBuilder()
    b.width, b.height = 120, 120
    self.draw_avatar(MapCord(60, 60), b, size=110, layer="avatar")

    # Piece icon (pip) and flag banner
    pip = self.icon(size=48)
    pip = Div(NotStr(self.flag.pieceLogo(piece_type=self.type,size=48)))
    #flag_b = self.flag.flag_banner(width=24, height=24, wavy=False)

    flag_b = self.flag.logo(
        size=48, animal=self.animal,
        piece_id=f"sq_{self.name}")


    primary = self.flag.primary
    secondary = self.flag.comp
    light = self.flag.lightComp

    if self.place is not None and self.place.parent is not None:
        parts = self.place.parent 
        place = self.place
        overlays = (
            CreamOverlay(stylized=True),
            RiverOverlay(max_width=4),
            LanternOverlay(),
            PieceOverlay(),
            FogOverlay(),
            HexNumberOverlay(),
        )
        index = parts.hp2i(place.location)
        nearby = set(parts.grid.indices_in_range(index, 1))
        region = HexRegion(nearby, parts.grid)
        ctx = parts.overlayContext(region=region, radius=20)
   
        #ctx = self.place.parent.overlayContext()
        middle = self.place.parent.place_ft(self.place,*overlays, distance=1,ctx=ctx)
        middleStyle = f"""
            position:relative; height:260px; overflow:hidden;
            """
        year_badge = Div("")
        textO = 0
    else:
        b = SVGBuilder()
        b.width, b.height = 120, 120
        self.draw_avatar(MapCord(60, 60), b, size=110, layer="avatar")
        middle = Div(NotStr(b.xml()), style="position:absolute; bottom:10px; left:50%; transform:translateX(-50%); z-index:2;")
        middleStyle = f"""
            position:relative; height:220px; overflow:hidden;
            background:linear-gradient(to bottom,{self.flag.lightPrimary},{self.flag.darkPrimary});
            clip-path:polygon(0 0, 100% 0, 100% 90%, 50% 100%, 0 90%);
        """
        year_badge = Div(
        str(self.year),
        style=f"""
            position:absolute; bottom:0; left:50%; transform:translate(-50%,50%);
            width:40px; height:40px; border-radius:12px;
            background:linear-gradient(to bottom,{light},{secondary});
            color:white; font-weight:800; font-size:14px;
            display:flex; align-items:center; justify-content:center;
            z-index:10;
        """)
        textO = 0.35

    # Year badge at bottom-center of image area
    

    # Top-left circle with piece icon
    tl_circle = Div(
        Div(pip, style="position:absolute; right:8px; bottom:8px;"),
        style=f"""
            position:absolute; top:0; left:0; transform:translate(-50%,-50%);
            width:64px; height:64px; border-radius:50%;
            border:1px solid #e5e5e5; background:white;
            z-index:10;
        """)

    # Top-right circle with flag
    tr_circle = Div(
        Div(NotStr(flag_b), style="position:absolute; left:8px; bottom:8px;"),
        style=f"""
            position:absolute; top:0; right:0; transform:translate(50%,-50%);
            width:64px; height:64px; border-radius:50%;
            border:1px solid #e5e5e5; background:white;
            z-index:10;
        """)

    # Faded name behind avatar
    bg_text = Div(
        Div(self.type.value.title()),
        Div(self.name.split()[0] if self.name else ""),
        
        style=f"""
            position:absolute; top:20px; left:50%; transform:translateX(-50%);
            text-align:center; font-size:48px; line-height:0.85em;
            font-weight:800; text-transform:uppercase; font-style:italic;
            color:white; opacity:{textO}; mix-blend-mode:overlay;
            pointer-events:none; z-index:1; letter-spacing:-0.05em;
        """)

    # Image area with gradient + chevron clip
    
    image_area = Div(
        bg_text,
        middle,
        style=middleStyle)


    # Faded name behind avatar
    bg_text = Div(
        Div(self.type.value.title()),
        Div(self.name.split()[0] if self.name else ""),
        
        style=f"""
            position:absolute; top:20px; left:50%; transform:translateX(-50%);
            text-align:center; font-size:48px; line-height:0.85em;
            font-weight:800; text-transform:uppercase; font-style:italic;
            color:white; opacity:{textO}; mix-blend-mode:overlay;
            pointer-events:none; z-index:1; letter-spacing:-0.05em;
        """)
    #zDiv(self.countryName),
    # Inner card: image + badges + name
    
    inner = Div(
        Div(
            image_area, year_badge, tl_circle, tr_circle,
            style="position:relative; overflow:visible; padding-bottom:16px;"
        ),
        Div(
            H4(self.name, style="margin:0; font-size:18px; font-weight:700; letter-spacing:-0.02em;"),
            P(self.countryName, cls=TextPresets.muted_sm),
            style="text-align:center; padding:16px 0 8px 0;"
        ),
        style=f"""
            border-radius:20px; background:white; padding:16px;
            box-shadow:0 0 0 1px #e5e5e5;
        """)

    # Stats row
    stats = [
        ("Food", str(self.food.diet)),
        ("Sight", str(self.food.bonus_sight)),
        ("Moves", str(self.food.drain)),
    ]
    stat_divs = [Div(
        Div(val, style="font-size:14px; font-weight:700; margin-bottom:4px;"),
        Div(label, style="font-size:10px; text-transform:uppercase;"),
        style="text-align:center; padding:0 20px; color:white;"
    ) for label, val in stats]

    stats_row = Div(
        *stat_divs,
        style="display:flex; justify-content:center; gap:0; padding:16px 0; "
              "divide-x divide-gray-200;"
    )

    # Outer card
    light = self.flag.tri1
    return Div(
        Div(inner, stats_row,
            style=f"border-radius:24px; border:1px solid #e5e5e5; background:{light};"),
        style="width:280px; padding:0 20px; margin:8px auto;"
    )


In [ ]:
myStuff.loadSquads()

ourTroop = myStuff.squads[0]
theKing = ourTroop.pieces[-1]
show(theKing)

In [ ]:
theKing.place

In [ ]:
show(ourTroop)


It seems like squad _ft_ is cropped on the left

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The issue is that the corner pip circles bleed outside the card bounds with `transform:translate(-50%,-50%)`, and the carousel's `overflow-x:auto` clips them on the left (and top). Fix it by adding padding to the carousel to give those circles room:

```python
carousel = Div(
    *slides,
    style="display:flex; flex-direction:row; overflow-x:auto; gap:16px; padding:40px 16px 16px 40px;"
)
```

The `40px` top padding accommodates the circles overflowing upward, and the `40px` left padding stops the first card's top-left circle from being clipped.

Now the headings like king x are now cropped

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The problem is that `overflow-x:auto` also implicitly clips vertically, cutting off the pip circles that bleed upward into the section header area. Fix it by separating the scroll container from the padding:

```python
carousel = Div(
    Div(
        *slides,
        style="display:flex; flex-direction:row; gap:16px; padding:0 16px 16px 40px;"
    ),
    style="overflow-x:auto; padding-top:40px;"
)
```

The outer div handles `padding-top` with `overflow:visible` (default), so the circles can bleed upward freely. The inner div scrolls horizontally without clipping the overflow. The section headers above won't be touched.

In [ ]:
ourTroop.table()

In [ ]:
aPiece = ourTroop.pieces[0]
show(aPiece)

In [ ]:
aPiece.icon()

In [ ]:
#| export
@patch
def orient_facings(self: Squad, parts: GameParts):
    """Cheap target-aware facing: blend ally connectivity with target projection."""
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2: return
    
    target = getattr(self, '_target', None)
    target_hp = parts.i2hp(target.hex_idx) if target and target.hex_idx >= 0 else None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    allies = {parts.hp2i(p.place.location) for p in placed}
    
    for piece in placed:
        my_hp = piece.place.location
        bonus = piece.food.bonus_sight
        eff = max(2, piece.place.level) + bonus
        
        best_dir, best_score = piece.place.facing, -999
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in my_hp.field_of_view(d, eff))
            
            # Connectivity: always valuable
            score = len((allies - {parts.hp2i(my_hp)}) & cone) * 3.0
            
            # King visibility bonus
            if king and king is not piece:
                if parts.hp2i(king.place.location) in cone:
                    score += 4.0
            
            # Target projection (the new part)
            if target_hp is not None:
                target_idx = parts.hp2i(target_hp)
                if target_idx in cone:
                    score += 8.0
                else:
                    # Partial: how far does cone extend toward target?
                    if cone:
                        min_dist = min(parts.i2hp(h).distance(target_hp) for h in cone)
                        score += 3.0 / max(1, min_dist)
            
            if score > best_score:
                best_score = score
                best_dir = d
        
        piece.place.facing = best_dir


In [ ]:
#| export
def LanternOverlay(**kw) -> OverlaySpec:
    """Render sight regions for placed pieces using their lantern style, plus facing bars."""
    def render(ctx) -> str:
        grid, builder = ctx.grid, ctx.builder
        N = len(grid.hexes)
        r = grid.radius
        parts = []

        for piece in ctx.pieces:
            if piece.place is None: continue
            builder.add_style(piece.lantern)

            # Sight region
            for idx in sorted(piece.sight().hexes):
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        h = grid.hexes[fi]
                        parts.append(Hex(h.radius, h.center, piece.lantern, v=h.v).svg())

            # Facing bar at piece's location
            loc_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(loc_idx):
                if 0 <= fi < N:
                    c = grid.hexes[fi].center
                    glyphs = DiagramGlyphs(piece.flag, size=r * 0.7)
                    glyphs.register_styles(builder)
                    dirs = HexPosition.directions()
                    facing_dir = dirs.index(piece.place.facing) if piece.place.facing in dirs else 0
                    parts.append(glyphs.facing_bar(c, facing_dir))

        return '\n'.join(parts)

    return OverlaySpec("lanterns", render, requires={'pieces'}, priority=62)



In [ ]:
#| export
def PieceOverlay(**kw) -> OverlaySpec:
    """Render all pieces using their flag's chess-piece SVG."""
    def render(ctx) -> str:
        if not ctx.pieces:
            return ""

        grid = ctx.grid
        N = len(grid.hexes)
        scale = grid.radius * 0.8 / 22.5
        parts = []

        for i, piece in enumerate(ctx.pieces):
            if piece.place is None or piece.flag is None:
                continue

            coarse_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(coarse_idx):
                if 0 <= fi < N:
                    center = grid.hexes[fi].center
                    pid = f"pc_{piece.id}_{i}"

                    svg_str, pat_def = piece.flag.piece_svg(
                        piece.type,
                        MapCord(center.x, center.y),
                        scale=scale,
                        size='board',
                        piece_id=pid,
                    )

                    if pat_def is not None:
                        ctx.builder.add_definition(pat_def)

                    parts.append(svg_str)

        return '\n'.join(parts)

    return OverlaySpec("pieces", render, requires={'pieces'}, priority=80)


In [ ]:
#| export
def SquadNamesOverlay(font_size: int = 14, font_family: str = "Cinzel", **kw) -> OverlaySpec:
    """Render squad names at the centroid of their placed pieces."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts = []
        
        for squad in ctx.squads:
            if not squad.name:
                continue
            
            # Collect fine-grid indices for all placed pieces
            placed_fine = []
            for piece in squad.pieces:
                if piece.place is None:
                    continue
                coarse_idx = piece.place.parent.hp2i(piece.place.location)
                for fi in ctx.fine_indices(coarse_idx):
                    if 0 <= fi < N:
                        placed_fine.append(fi)
            
            if not placed_fine:
                continue
            
            # Centroid of placed piece centers
            cx = sum(grid.hexes[fi].center.x for fi in placed_fine) / len(placed_fine)
            cy = sum(grid.hexes[fi].center.y for fi in placed_fine) / len(placed_fine)
            
            # Label style from flag
            label_style = squad.flag.labelStyle(f"sqname_{squad.name.replace(' ', '_')}")
            ctx.builder.add_style(label_style)
            ctx.builder.add_font(font_family)
            
            # Offset below centroid so it doesn't sit on top of pieces
            label_y = cy + grid.radius * 1.8
            
            parts.append(
                f'<text x="{cx:.1f}" y="{label_y:.1f}" text-anchor="middle" '
                f'font-size="{font_size}" font-family="\'{font_family}\', sans-serif" '
                f'dominant-baseline="middle" class="{label_style.name}">'
                f'{squad.name}</text>'
            )
        
        return '\n'.join(parts)
    
    return OverlaySpec("squad_names", render, requires={'squads'}, priority=90)


In [ ]:
myStuff = GameParts()

In [ ]:
oasis_idxs = myStuff.fy.find_oasis(n=len(myStuff.squads))

In [ ]:
showDemo = True

In [ ]:
ctx = myStuff.overlayContext()
if showDemo:
    TerrainDisplay(
        CreamOverlay(stylized=True),
        RiverOverlay(max_width=4),
        PieceOverlay(),
        OceanWaveOverlay(
        num_waves=5,
        spacing=8,
        amplitude_start=3.0,
        amplitude_decay=0.6,
        wavelength=40,
        stroke_color="#4a7fb5",
        opacity_start=0.5,
        opacity_decay=0.7,
        stroke_width=1.2),
        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
myStuff.loadSquads()
TerrainDisplay(
        CreamOverlay(stylized=True),
        RiverOverlay(max_width=4),
        PieceOverlay(),
        FogOverlay(),
        #OasisOverlay(),
        HexNumberOverlay(),
        OceanWaveOverlay(
        num_waves=5,
        spacing=8,
        amplitude_start=3.0,
        amplitude_decay=0.6,
        wavelength=40,
        stroke_color="#4a7fb5",
        opacity_start=0.5,
        opacity_decay=0.7,
        stroke_width=1.2),
        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
??GameParts.loadSquads

In [ ]:
@patch
def loaddebugSquads(self:GameParts, year: int=1880, seed: int = 0,levels=3):
    self.flags = CountryFlag.seaborn("bright",levels=levels)
    oasis_idxs = myStuff.fy.find_oasis(n=len(myStuff.squads))
    self.squads = []
    for i,flag in enumerate(self.flags):
        roster = {
            PieceType.PAWN: 3, 
            #PieceType.BISHOP: 2, 
            #PieceType.QUEEN: 2, 
            PieceType.KING: 1, 
            #PieceType.KNIGHT: 4
        }
        pieceKinds = [pt for pt, n in roster.items() for _ in range(n)]


        squad = flag.make_squad(pieceKinds, year=year, seed =seed) 
        kingPlace = self.location(oasis_idxs[x],facing=HexDirection.NORTH)
        squad.pieces[-1].place = kingPlace
        self.squads.append(squad)

In [ ]:
myStuff.loadSquads()